# Prompt 08: Advanced Techniques

1. ReAct-style prompt with manual tool loop
2. Self-refine loop (draft → critique → revise)
3. Decompose-solve-recompose on a multi-part question
4. Parallel sampling + judge-based best-of-N
5. Exercises: iterative deepening, confirm-before-action

Deps: `pip install anthropic openai`

In [ ]:
import os, re, json

def call(system, user, max_tokens=600, temperature=0):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(model='claude-sonnet-4-6', max_tokens=max_tokens, temperature=temperature,
            system=system, messages=[{'role':'user','content':user}])
        return r.content[0].text
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model='gpt-4o-mini', max_tokens=max_tokens, temperature=temperature,
            messages=[{'role':'system','content':system},{'role':'user','content':user}])
        return r.choices[0].message.content
    return '[no provider]'

## 1. ReAct with a Mock Tool

In [ ]:
def kb_lookup(query):
    # Mock tool. Real version would call a vector DB or API.
    kb = {
        'pto': 'Full-time employees accrue 15 days PTO per year. [KB-PTO-1]',
        'rollover': 'PTO may roll over up to 5 days. [KB-PTO-2]',
        'unpaid': 'PTO does not accrue during unpaid leave. [KB-PTO-3]',
    }
    hits = [v for k, v in kb.items() if k in query.lower()]
    return '\n'.join(hits) if hits else 'No matches.'

REACT_SYSTEM = '''You solve user questions by alternating Thought, Action, and Observation.

Available tools:
- kb_lookup(query: str): search the HR knowledge base.

Format strictly:
Thought: <your reasoning>
Action: kb_lookup("<search query>")

Wait for an Observation: from the system.
After enough Observations, produce:
Final Answer: <your answer citing [KB-*] ids>.'''

question = 'If I take 4 weeks of unpaid leave, how much PTO will I accrue this year?'
transcript = f'Q: {question}\n'

for _ in range(5):
    step = call(REACT_SYSTEM, transcript, max_tokens=300)
    transcript += step + '\n'
    print(step)
    if 'Final Answer:' in step:
        break
    m = re.search(r'Action:\s*kb_lookup\(\s*["\']([^"\']+)["\']\s*\)', step)
    if not m: break
    obs = kb_lookup(m.group(1))
    transcript += f'Observation: {obs}\n'
    print(f'Observation: {obs}\n')

## 2. Self-Refine Loop

In [ ]:
task = 'Write a function `merge_sorted_lists(a, b)` in Python that merges two sorted integer lists into one sorted list.'

draft = call('You are a Python programmer.', task + ' Return only code.')
print('=== DRAFT ===\n', draft)

critique = call(
    'You are a strict code reviewer. Identify bugs, edge cases missed, or style issues. Be concise.',
    f'Task: {task}\n\nCode:\n{draft}\n\nReview.'
)
print('\n=== CRITIQUE ===\n', critique)

revised = call(
    'You are a Python programmer. Apply the reviewer\'s feedback. Return only the revised code.',
    f'Task: {task}\n\nOriginal:\n{draft}\n\nFeedback:\n{critique}\n\nRevised:'
)
print('\n=== REVISED ===\n', revised)

## 3. Decompose-Solve-Recompose

In [ ]:
big_q = 'Compare Python and Go for building a high-throughput web scraper. What are the tradeoffs, and which would you pick if you must parse JSON deeply and run 200 concurrent fetches?'

# Step 1: decompose
subs_raw = call('You break hard questions into 3-4 focused sub-questions.',
                f'Break into sub-questions:\n{big_q}\n\nReturn a numbered list only.')
print('SUBS:\n', subs_raw)

subs = [re.sub(r'^\d+\.\s*','', s).strip() for s in subs_raw.split('\n') if re.match(r'^\d+\.', s.strip())]

# Step 2: solve each
answers = []
for s in subs[:4]:
    a = call('Answer in 2 sentences, no fluff.', s)
    answers.append((s, a))
    print(f'\nQ: {s}\nA: {a}')

# Step 3: recompose
packed = '\n'.join(f'- {q}\n  -> {a}' for q, a in answers)
final = call('Synthesize into a coherent answer using the sub-answers.',
             f'Main question: {big_q}\n\nSub-answers:\n{packed}\n\nSynthesize:')
print('\n=== SYNTHESIS ===\n', final)

## 4. Parallel Best-of-N with Judge

In [ ]:
prompt = 'Write a tagline for a productivity app aimed at remote software teams. Max 10 words.'

candidates = [call('You are a creative copywriter.', prompt, temperature=1.0) for _ in range(5)]
for i, c in enumerate(candidates): print(f'{i}:', c.strip())

judge_prompt = 'Rank these taglines 1-5 (best first) and return just the ordered indices like [3,1,4,2,0].\n\n' \
             + '\n'.join(f'{i}: {c.strip()}' for i, c in enumerate(candidates))

ranking = call('You are a strict creative judge.', judge_prompt)
print('\nranking:', ranking)

m = re.search(r'\[([0-9,\s]+)\]', ranking)
if m:
    best = int(m.group(1).split(',')[0])
    print(f'\nbest (#{best}):', candidates[best].strip())

## 5. Exercise: Iterative Deepening on a Design Question

For: *"Design a vector-search service for 500M documents."*

- Turn 1: 5 candidate approaches, 1 line each.
- Turn 2: pick 2; for each, give 3 pros and 3 cons.
- Turn 3: pick 1; sketch architecture diagram in ASCII.
- Turn 4: identify top 3 operational risks of the chosen approach.

Compare to asking for the full design in one shot. Measure which you find more useful (and which cost more tokens).

## 6. Exercise: Confirm-Before-Action Agent

Build an agent prompt that:
1. Proposes an action (e.g., `send_email(to='alice@corp.com', subject='...', body='...')`).
2. Requests user confirmation.
3. Only 'executes' the mock action on confirmation.

This is the seed of safe agent design — any mutating tool call goes through a confirmation gate.

## 7. Takeaways

- **ReAct** = Thought + Action + Observation loop; native tool use is its prod form.
- **Self-refine** works on code and safety-critical outputs.
- **Decompose** when sub-questions are cleanly separable.
- **Best-of-N + judge** trades N× cost for quality.
- **Gate mutating actions** on human confirmation.

Next: **09 — Prompt Optimization** — DSPy, APE, and automated prompt search.